In [1]:
import pandas as pd

In [2]:
import torch

In [3]:
from torch import nn, optim

In [4]:
from string import punctuation

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
df = pd.read_csv("C:\\git\\dados\\nlp\\news_sentiment_analysis.csv", encoding="utf-8")

In [8]:
df["sentimento_numerico"] = df["Sentiment"].map( {"positive": 1, "negative": 0, "neutral": 0} )

In [9]:
df = df[["Description", "Sentiment", "sentimento_numerico"]]

In [10]:
def limpar_texto( texto : str ) -> str:
    trans_table = str.maketrans("—", " ", punctuation)
    texto_limpo = texto.lower().translate( trans_table ).replace("  ", " ")
    # texto_limpo.replace("  ", " ")
    return texto_limpo

In [11]:
df["TextoLimpo"] = df["Description"].apply( limpar_texto )
df

,Description,Sentiment,sentimento_numerico,TextoLimpo
0,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,1,st george kaitlyn larson a firstyear teacher ...
1,"Harare, Zimbabwe – Local businesses are grappl...",neutral,0,harare zimbabwe – local businesses are grappli...
2,(marketscreener.com) Billionaire Elon Musk has...,positive,1,marketscreenercom billionaire elon musk has do...
3,(marketscreener.com) A U.S. trade regulator on...,negative,0,marketscreenercom a us trade regulator on frid...
4,4.5 million households in the U.S. have solar ...,positive,1,45 million households in the us have solar pan...
...,...,...,...,...
3495,QRG Capital Management Inc. increased its stak...,positive,1,qrg capital management inc increased its stake...
3496,QRG Capital Management Inc. bought a new posit...,positive,1,qrg capital management inc bought a new positi...
3497,QRG Capital Management Inc. boosted its stake ...,positive,1,qrg capital management inc boosted its stake i...
3498,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,0,westford mass july 18 2024 prnewswire accordin...


In [12]:
MAX_PALAVRAS = 3000

In [13]:
vetorizador = CountVectorizer(max_features = MAX_PALAVRAS)

In [14]:
texto_vetorizado = vetorizador.fit_transform( df["TextoLimpo"] )
texto_vetorizado

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 98285 stored elements and shape (3500, 3000)>

In [15]:
X = torch.tensor( texto_vetorizado.toarray(), dtype=torch.float32)
X

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [16]:
Y = torch.tensor(df["sentimento_numerico"], dtype=torch.float32)
Y = Y.reshape(3500, 1)

In [17]:
vocabulario = vetorizador.get_feature_names_out()

In [18]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.20, random_state = 100)

In [19]:
modelo = nn.Sequential( 
    nn.Linear(in_features=MAX_PALAVRAS, out_features=1),
    #nn.Linear(in_features=MAX_PALAVRAS, out_features=512),
    #nn.Linear(in_features=512, out_features=1),
    nn.Sigmoid()
)
modelo

Sequential(
  (0): Linear(in_features=3000, out_features=1, bias=True)
  (1): Sigmoid()
)

In [20]:
for param in modelo.parameters():
    print(param)

Parameter containing:
tensor([[ 0.0044,  0.0089, -0.0123,  ...,  0.0177,  0.0126,  0.0174]],
       requires_grad=True)
Parameter containing:
tensor([0.0112], requires_grad=True)


In [21]:
funcao_perda = nn.BCELoss()

In [22]:
otimizador = optim.SGD( modelo.parameters(), lr = 0.1)

In [23]:
epocas = 5000
for epoca in range(epocas):
    y_hat = modelo( X_train )
    loss = funcao_perda( y_hat, Y_train )
    if epoca % 100 == 0: 
        print("Loss: ", loss)
    loss.backward()
    otimizador.step()

Loss:  tensor(0.6933, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.1720, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0801, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0335, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0117, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0104, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0062, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0050, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0037, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0029, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0028, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0025, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0019, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0013, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0011, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0006, grad_fn=<BinaryCrossEntropyBackward0>)
Loss:  tensor(0.0007, gr

In [24]:
# acertos = 0 

In [25]:
# texto = df["TextoLimpo"].loc[3496]


In [26]:
# teste_vetorizador = CountVectorizer(vocabulary = vocabulario)

In [27]:
# X_test = teste_vetorizador.fit_transform( df["TextoLimpo"] )
# X_test.toarray()

In [28]:
# teste_vetorizador.get_feature_names_out()

In [29]:
# X_test = torch.tensor( X_test.toarray(), dtype=torch.float32 ) 
# X_test.shape

In [30]:
y_hat_test = modelo( X_test )

In [31]:
acertos = 0
for idx, y_hat in enumerate(y_hat_test):
    resultado_oficial = Y_test[idx].item()
    resultado_previsto = 0
    if y_hat.item() >= 0.5:
        resultado_previsto = 1
    # print(f"{resultado_previsto} == {resultado_oficial} => {resultado_previsto == resultado_oficial}")
    if resultado_previsto == resultado_oficial:
        acertos = acertos + 1
print("Porcentagem de acerto: ",  (acertos / len(y_hat_test)) * 100 )    

Porcentagem de acerto:  90.85714285714286
